# Agriculature Water Use estimation from observed ET

Pywatershed provides the agriculture irrigation use estimation capabilities previously implemented into the PRMS-only mode of GSFLOW (Regan et al., 2023). The version of GSFLOW reproduced by pywatershed is (the currenly unreleased) 2.4.0. The pywatershed application demonstrated in this notebook is drawn from the data release of Martin et al. (2023), wherein GSFLOW 2.3 was used estimate CONUS agricultural irrigation water.
Martin et al (2023) used actual evapotransipration observations/estimates from the Simplified Surface Energy Balance (SSEBop, Senay and others, 2013; Senay and others, 2020) executed in the OpenET (Melton and others, 2021) web-based application implemented in Google Earth Engine. Estimates provided by OpenET/SSEBop were summarized to hydrologic response units (HRUs) in the National Hydrologic Model (NHM; Regan and others, 2019). Irrigated lands for the CONUS were provided by the Landsat-based Irrigation Dataset (LANID; Xie and others, 2019) for each year of the reanalysis period. 

Pywatershed implements the classes `PRMSRunoffAg`, `PRMSSoilzoneAg`, and `PRMSSoilzoneAgObsET` to reproduce agriculture water use estimates. The first feature of these classes is dual area soil moisture accounting. In these classes, an HRU consists of impervious, pervious, and agricultural areas. The pervious and agricultural areas have independent soil moisture accounting and parameters and the pervious area implementation is the same as standard PRMSSoilzone. `PRMSRunoffAg` calculates `infil_ag`, an input to the new soilzone classes. 

The class `PRMSSoilzoneAgObsEt` extends `PRMSSoilzoneAg` to include the second major innovation, iterative matching of "observed" (or externally specified) actual evapotranspiration (AET). Agricultural irrigation is estimated as modeled error in observed AET on the agricultural area. An iteration loop (over all HRUs) adjusts the amount of applied irrigation to reduce the difference of observed AET and agricultural AET. The iteration terminates when one of the following conditions is met: a) the AET error drops below a threshold, b) soil moisture is above some threshold (ag_soilwater_defecit_min), or c) a maximum number of iterations is reached. Each pass through the iteration loop resets the agricultural soil moisture state variables to their values at the start current of the timestep and adds the cumulative AET error (in each HRU) over all previous iterations as irrigation, or extra infiltration, to the agricultural fraction of the HRU.

Below, we'll use pywatershed and the observed actual evapotranspiration data and the ag fraction data to show how adding the agricultrue fraction, with `PRMSSoilzoneAg`, and also estimating agriculture water use, with `PRMSSoilzoneAgObsET`, differ from running the NHM configuration. 

References
-----------

> Melton, F. S., Huntington, J., Grimm, R., Herring, J., Hall, M., Rollison, D., Erickson, T., et al., 2021, Openet: Filling a critical data gap in water management for the western United States: JAWRA Journal of the American Water Resources Association. [https://doi.org/10.1111/1752-1688.12956](https://doi.org/10.1111/1752-1688.12956)
> 
> Regan, R. S., Niswonger, R., Morway, E., Triana, E., 2023, Version 2.3.0 of Coupled Ground-Water and Surface-Water Flow Model Based on the Integration of the Precipitation-Runoff Modeling System (PRMS) and the Modular Ground-Water Flow Model. [https://doi.org/10.5066/p9uy8g6l](https://doi.org/10.5066/p9uy8g6l)
> 
> Senay, G. B., Kagone, S., & Velpuri, N. M., 2020, Operational global actual evapotranspiration: Development, Evaluation, and Dissemination: Sensors 2020 20(7) 1915. [https://doi.org/10.3390/s20071915](https://doi.org/10.3390/s20071915)
>
> Senay, G. B., Bohms, S., Singh, R. K., Gowda, P. H., Velpuri, N. M., Alemu, H., & Verdin, J. P., 2013, Operational evapotranspiration mapping using remote sensing and weather datasets: A new parameterization for the SSEB approach: JAWRA Journal of the American Water Resources Association 49(3), 577-591. [https://onlinelibrary.wiley.com/doi/10.1111/jawr.12057](https://onlinelibrary.wiley.com/doi/10.1111/jawr.12057)
>
> Xie, Y., Lark, T. J., Brown, J. F., & Gibbs, H. K., 2019, Mapping irrigated cropland extent across the conterminous United States at 30 m resolution using a semi-automatic training approach on Google Earth Engine: ISPRS Journal of Photogrammetry and Remote Sensing 155, 136-149. [https://doi.org/10.1016/j.isprsjprs.2019.07.005](https://doi.org/10.1016/j.isprsjprs.2019.07.005)



In [ ]:
import pathlib as pl
import warnings

import jupyter_black
import xarray as xr

import pywatershed as pws

jupyter_black.load()

warnings.filterwarnings(action="ignore", message=r"datetime.datetime.utcnow")
pws.utils.addtl_domain_files.download()  # may need force=True argument passed

In [ ]:
run_dir_exist_fail = False
pkg_root = pws.constants.__pywatershed_root__
domain_dir = pkg_root / "data/pywatershed_addtl_domains/fgr_ag_2yr"
notebook_dir = pl.Path("./ag_runs_fgr_ag_2yr")
if not notebook_dir.exists():
    notebook_dir.mkdir()

## Experimental design

1. "NHM": NHM configuration
2. "OL": Open loop agriculture implemetation without AET observations
3. "Analysis": Agriculture implemetation with AET observations to get irrigation analysis/estimates

In [ ]:
nhm_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoff,
    pws.PRMSSoilzone,
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

wu_ol_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoffAg,
    pws.PRMSSoilzoneAg,
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

wu_analysis_processes = [
    pws.PRMSSolarGeometry,
    pws.PRMSAtmosphere,
    pws.PRMSCanopy,
    pws.PRMSSnow,
    pws.PRMSRunoffAg,
    pws.PRMSSoilzoneAgObsET,
    pws.PRMSGroundwater,
    pws.PRMSChannel,
]

Wrangle the various input files. Note that we are running model configuration which ignores potential input data in the domain directory. 

In [ ]:
control_spinup_file = domain_dir / "spinup.control"
control_analysis_file = domain_dir / "analysis.control"

control_nhm = pws.Control.load_prms(control_analysis_file)
control_ol = pws.Control.load_prms(control_analysis_file)
control_analysis = pws.Control.load_prms(control_analysis_file)

control_nhm.options["input_dir"] = domain_dir
control_ol.options["input_dir"] = domain_dir
control_analysis.options["input_dir"] = domain_dir

# For the NHM configuration, this dosent matter but for the ol
# using PRMSSoilzoneAg (and not PRMSSoilzoneAgObsET) it does matter
control_nhm.options["iter_aet_flag"] = False
control_ol.options["iter_aet_flag"] = False

control_nhm.options["netcdf_output_dir"] = notebook_dir / "run_nhm"
control_ol.options["netcdf_output_dir"] = notebook_dir / "run_ol"
control_analysis.options["netcdf_output_dir"] = notebook_dir / "run_analysis"

control_nhm.options["netcdf_output_var_names"] = [
    "hru_ppt",
    "hru_actet",
    "perv_actet",
    "potet",
    "soil_moist",
    "soil_rechr",
]
control_ol.options["netcdf_output_var_names"] = [
    "hru_ppt",
    "hru_actet",
    "perv_actet",
    "hru_ag_actet",
    "potet",
    "soil_moist",
    "ag_soil_moist",
    "ag_soil_moist_prev",
    "soil_rechr",
    "ag_soil_rechr",
    "ag_irrigation_add",
]
control_analysis.options["netcdf_output_var_names"] = control_ol.options[
    "netcdf_output_var_names"
]

# document what this is about and when we need it.
control_nhm.options["intcp_changeover_in_net_rain"] = False
control_ol.options["intcp_changeover_in_net_rain"] = False
control_analysis.options["intcp_changeover_in_net_rain"] = False

Build a helper function to create run directories. 

In [ ]:
def mk_run_dirs(control_list: list, exist_fail=True) -> bool:
    for cc in control_list:
        if (thedir := cc.options["netcdf_output_dir"]).exists():
            if exist_fail:
                raise ValueError(
                    f"{thedir} directory can not exist before running this notebook."
                )
            else:
                print(f"Run directory {thedir} exists, skipping")
                return False
        else:
            thedir.mkdir()
            return True

## NHM run

Note that we'll use PRMS-legacy style model instantiation in all the cases here for brevity.

In [ ]:
if mk_run_dirs(
    [
        control_nhm,
    ],
    exist_fail=run_dir_exist_fail,
):
    params_nhm = pws.parameters.PrmsParameters.load(
        domain_dir / control_nhm.options["parameter_file"]
    )
    model_nhm = pws.Model(
        nhm_processes, control=control_nhm, parameters=params_nhm
    )
    model_nhm.run(finalize=True)

## Open Loop Run

The PRMSRunoffAg, PRMSSoilzoneAg, and PRMSSoilzoneAgObsET have a an "ag_frac" input. This input is required by pywatershed in either case, if the ag frac is dynamic or static, where as in PRMS a default ag_frac would be found in the parameter file is a dynamic one was not requested. In PRMS-legacy mode, we must create this input file as a netcdf file (is that true, cant we pass the dynamic parameter file?). If instantiating our Model the pywatershed way, we have additional options. Following a PRMS-legacy instantiation, we can easily re-use an existing input files with xarray to define the a time-varying ag_frac input. 

In [ ]:
if mk_run_dirs(
    [
        control_ol,
    ],
    exist_fail=run_dir_exist_fail,
):
    params_spinup_file = control_spinup_file.parent / control_ol.options.get(
        "parameter_file"
    )
    params_nhm = pws.parameters.PrmsParameters.load(params_spinup_file)
    params_spinup = pws.parameters.PrmsParameters.load(params_spinup_file)
    ag_frac = xr.load_dataarray(domain_dir / "tmin.nc")
    ag_frac[:, :] = params_spinup.parameters["ag_frac"]
    ag_frac.name = "ag_frac"
    ag_frac_file = domain_dir / "ag_frac.nc"
    if not ag_frac_file.exists():
        # I probably should NOT write this file here. SHOULD DELETE IT
        ag_frac.to_netcdf(ag_frac_file)
    params_ol = pws.parameters.PrmsParameters.load(
        control_spinup_file.parent / control_ol.options.get("parameter_file")
    )

    model_ol = pws.Model(
        wu_ol_processes, control=control_ol, parameters=params_ol
    )
    model_ol.run(finalize=True)

## Analysis Run

In [ ]:
if mk_run_dirs(
    [
        control_analysis,
    ],
    exist_fail=run_dir_exist_fail,
):
    params_analysis = pws.parameters.PrmsParameters.load(
        control_analysis_file.parent
        / control_analysis.options.get("parameter_file")
    )

    model_analysis = pws.Model(
        wu_analysis_processes,
        control=control_analysis,
        parameters=params_analysis,
    )
    model_analysis.run(finalize=True)

## Comparisons

In [ ]:
shp_file = "../pywatershed/data/pywatershed_gis/fgr_2yr/model_nhru.shp"
run_colors = {
    "NHM": "#8da0cb",
    "OL": "#66c2a5",
    "Analysis": "#fc8d62",
}
comparer = pws.analysis.HRUComparisonPanel(
    shapefile_path=shp_file,
    variable_names=control_analysis.options["netcdf_output_var_names"]
    + ["ag_frac", "aet_observed", "pet_observed"],
    run_directories={
        "NHM": control_nhm.options["netcdf_output_dir"],
        "OL": control_ol.options["netcdf_output_dir"],
        "Analysis": control_analysis.options["netcdf_output_dir"],
    },
    input_directories={
        "NHM": control_nhm.options["input_dir"],
        "OL": control_ol.options["input_dir"],
        "Analysis": control_analysis.options["input_dir"],
    },
    hru_id_column="nhm_id",
    # simplify_tolerance=500,  # detail of HRU polygons dialed w/ this parameter
)
app = comparer.create_app()
# Starting a separate panel view is more stable than trying to render in jupyter
app.show()  # start a separte panel viewer

The above code spawns an HRUComparisonPanel object in a new browser tab or window. For the requested variable names, the viewer allows by-HRU comparison of variables. It provides a spatial map, which shows either the temporal statistic of a single run or the difference between temporal statistics for any two selected runs. When an HRU is selected, it provides timeseries of the current variable for all runs in a single plot below. The panel let's us quickly get an idea of what we are looking at because it has many options and quick access to all the runs. Once we use it it to get oriented, we'll see next that we can use the HRUComparisonPanel object to create even more customized plots. First, we'll look at several screenshots of a potential exploration that can be followed with that viewer. 

To focus on "water use" or "agricultural irrigation" right off the bat, we will select the variable of interest for this purpose "ag_irrigation_add". First, in the panel results below, note the distribution of irrigated areas across this domain. We will zoom in on several of these to get acquainted with the results. 

![Farmington, NM, panel](static/big_sandy_panel_ag_irrigation_add.png)

### HRU 84966 - Farmington, NM
Let's first focus on one of the HRUs with a value towards the higher end of the range found across the map. This HRU, called out on the map, has nhm_id 84966 and is located just south of Farmington, NM. In the timeseries plot above, we can see the repeating pattern that from about late May into October, irrigation is applied. We also see some anomalous large irrigation applications at the beginning of the timeseries, perhaps soil moisture defecits from lack of irrigation being met??

We'll zoom in to see its location more clearly. And we'll take a satellite view of the location to get a sense of if there is indeed agriculture present.


| Location View | Satellite View |
|:---:|:---:|
| ![Farmington, NM, political](static/big_sandy_map_political.png) | ![Farmington, NM, satellite](static/big_sandy_map_satellite.png) |

By selecting different variables, we can get some sense of the hydrology and agriculture in this HRU. We can individually look at all the variables shown in the following plots. 


| |
|:---:|
| ![](static/big_sandy_ag_irrigation_add.png) |
| ![](static/big_sandy_potet.png) |
| ![](static/big_sandy_pet_observed.png) |
| ![](static/big_sandy_aet_observed.png) |
| ![](static/big_sandy_hru_ppt.png) |
| ![](static/big_sandy_hru_actet.png) |
| ![](static/big_sandy_hru_ag_actet.png) |
| ![](static/big_sandy_perv_actet.png) |
| ![](static/big_sandy_ag_soil_moist.png) |
| ![](static/big_sandy_ag_soil_moist_prev.png) |


We see quickly that this is a water-limited HRU (and region) and that actual ET in the model is governed by the amount of precipitation available. The modeled potential ET appears a decent match to the "observed" (OpenET) values, but the actual ET is very different. The comparison of the actual ET from the 3 runs shows that while including an agriculture soilzone in the OL run helps mitigate some of the extreme "flashy" response of actual ET to rainfaill seen in the NHM run, the missing process of irrigation is essential to correctly modeling the observed actual ET in this region. The elevated observed actual ET from roughly April through October, in boths years, is a signatures of agriculture in this HRU and region. We can see that estimating agricultural irrigation use from the observations helps simulate actual ET curves which are more realistic. Using these observations in the model also appears to help with overestimating actual ET peaks outside the growing/irrigation season. However, these quantities are in separate plots. In a moment, we'll use additional features of the HRUComparisonPanel object to make clear comparisons of these various quantities. 

Another notable feature of the plots is that two large initial irrigation pulses are applied at the start of the run in January, which is unusual timing. This is the system response to the use of AET observations in the analysis. The OL and the Analysis run have identical states at the start of the plotted period (2000-2001), but we can see that their soil moisture levels diverge sharply as a result of needing to match observed AET. If we plot ag_soil_moist_prev, we see that the divergence is immediate. 

Let us make cleaner comparisons of modeled and observed quantities. Let's start with modeled and observed potential ET. 

In [ ]:
comparer.selected_hru_widget.value = 86247

plt_width = 1300
plt_height = 400


# Plot different variables from different runs
def plot_potet():
    plot = comparer.hru_plot_together(
        {
            "Analysis": ["pet_observed", "potet"],
        },
        # hru_id=84966,  # optional - uses current selection if omitted
        width=plt_width,
        height=plt_height,
        title="Potential ET",
        renamer={
            "Analysis: pet_observed": "OpenET Potential ET",
            "Analysis: potet": "Modeled Potential ET",
        },
        colors={
            "OpenET Potential ET": "black",
            "Modeled Potential ET": "orange",
        },
    )
    display(plot)


plot_potet()

The modeled and "observed" potential ET are indeed very similar in overall magnitide and seasonal variation but als with respect to many individual fluctuations at shorter scales. We'll produce a similar but more detailed plot for actual ET.

In [ ]:
def plot_actet():
    plot = comparer.hru_plot_together(
        {
            "NHM": ["hru_actet"],
            "OL": ["hru_actet"],
            "Analysis": [
                "aet_observed",
                "hru_actet",
            ],
        },
        # hru_id=84966,  # optional - uses current selection if omitted
        title="Actual ET",
        width=plt_width,
        height=plt_height,
        renamer={
            "NHM: hru_actet": "NHM: Actual ET",
            "OL: hru_actet": "OL: Actual ET",
            "Analysis: hru_actet": "Analysis: Actual ET",
            "Analysis: aet_observed": "OpenET Actual ET",
        },
        colors={
            "OpenET Actual ET": "black",
            "NHM: Actual ET": run_colors["NHM"],
            "OL: Actual ET": run_colors["OL"],
            "Analysis: Actual ET": run_colors["Analysis"],
        },
    )
    display(plot)


plot_actet()

The plot shows that while the observed actual ET analysis dramatically imparts realism to our modeled results, the magnitude of the error reduction in the growing season is only roughly half the total error. 

ARE THERE IRRIGATION RATE LIMITS IMPOSED IN THE CODE?

We can include additional variables in the same plot (if they are in the same units). If we want to set the context for modeled actual ET in the broader conted of observed potential Et and available precipitation, we can easily add those variables. 

In [ ]:
# Plot different variables from different runs
def plot_actet_2():
    plot = comparer.hru_plot_together(
        {
            "NHM": ["hru_actet"],
            "OL": ["hru_actet"],
            "Analysis": [
                "aet_observed",
                "pet_observed",
                "hru_ppt",
                "hru_actet",
            ],
        },
        # hru_id=84966,  # optional - uses current selection if omitted
        title="Evapotranspiration & Precipitation",
        width=plt_width,
        height=plt_height,
        renamer={
            "NHM: hru_actet": "NHM: Actual ET",
            "OL: hru_actet": "OL: Actual ET",
            "Analysis: hru_actet": "Analysis: Actual ET",
            "Analysis: hru_ppt": "Modeled Precipitation",
            "Analysis: aet_observed": "OpenET Actual ET",
            "Analysis: pet_observed": "OpenET Potential ET",
        },
        colors={
            "Modeled Precipitation": "darkblue",
            "OpenET Potential ET": "gray",
            "OpenET Actual ET": "black",
            "NHM: Actual ET": run_colors["NHM"],
            "OL: Actual ET": run_colors["OL"],
            "Analysis: Actual ET": run_colors["Analysis"],
        },
    )
    display(plot)


plot_actet_2()

In [ ]:
# Plot different variables from different runs
def plot_actet_2():
    plot = comparer.hru_plot_together(
        {
            "NHM": ["hru_actet"],
            "OL": ["hru_actet"],
            "Analysis": [
                "aet_observed",
                "pet_observed",
                "hru_ppt",
                "hru_actet",
            ],
        },
        # hru_id=84966,  # optional - uses current selection if omitted
        title="Evapotranspiration & Precipitation",
        width=plt_width,
        height=plt_height,
        renamer={
            "NHM: hru_actet": "NHM: Actual ET",
            "OL: hru_actet": "OL: Actual ET",
            "Analysis: hru_actet": "Analysis: Actual ET",
            "Analysis: hru_ppt": "Modeled Precipitation",
            "Analysis: aet_observed": "OpenET Actual ET",
            "Analysis: pet_observed": "OpenET Potential ET",
        },
        colors={
            "Modeled Precipitation": "darkblue",
            "OpenET Potential ET": "gray",
            "OpenET Actual ET": "black",
            "NHM: Actual ET": run_colors["NHM"],
            "OL: Actual ET": run_colors["OL"],
            "Analysis: Actual ET": run_colors["Analysis"],
        },
    )
    display(plot)


plot_actet_2()

In [ ]:
comparer.selected_hru_widget.value = 86113
plot_potet()
plot_actet()
plot_actet_2()